In [4]:
import medim
import numpy as np

from medim_infer import create_gt_arr
from medim_infer import sam_model_infer
from medim_infer import data_postprocess
from medim_infer import data_preprocess
import os.path as osp
from collections import defaultdict

In [ ]:
clicks = [
    {
        'fg': np.array([[12, 390, 142]]), 
        'bg': np.array([], dtype=np.int32).reshape(0, 3)  
    }
]

img_dict = {
    'imgs': img,
    'clicks': np.array(clicks, dtype=object)  
}

np.savez('/data/yin/SAM-Med3D/data/validation/sample_20/img/test.npz', **img_dict)

In [18]:
img = np.load('/mnt/aperto/yin/sammed_3D/sample_20/img/test_img.npz')['imgs']

clicks = [
    {
        'fg': {1:np.array([[12, 390, 142]])}, 
        'bg': np.array([], dtype=np.int32).reshape(0, 3)  
    },
]

In [28]:
clicks = [
    {
        'fg': [(10, 20, 30), (15, 25, 35)],
        'bg': [(5, 10, 15)]
    },
    {
        'fg': [(40, 50, 60)],
        'bg': [(35, 45, 55), (30, 40, 50)]
    }
]


In [29]:
def read_data(img, clicks):

    # spacing = sitk_spacing
    spacing = [1.5, 1.5, 1.5]
    # z-score normalize imgs
    img = img.astype(np.float32)
    # parsing boxes/clicks tensor, allow category to has more than 1 clicks
    all_clicks = defaultdict(list)
    prev_pred = np.zeros_like(img, dtype=np.uint8)
    
    if (clicks is not None):
        for cls_idx, cls_click_dict in enumerate(clicks):
            for click in cls_click_dict['fg']:
            #     all_clicks[cls_idx].append(((click[2], click[1], click[0]), [1]))
                all_clicks[cls_idx].append((click, [1]))
            for click in cls_click_dict['bg']:
                all_clicks[cls_idx].append((click, [0]))

    return img, spacing, all_clicks, prev_pred

In [30]:
img, spacing, all_clicks, prev_pred = read_data(img, clicks)

In [31]:
for idx, cls_clicks in all_clicks.items():
    print(f"idx {idx} cls_clicks: {cls_clicks}")

idx 0 cls_clicks: [((10, 20, 30), [1]), ((15, 25, 35), [1]), ((5, 10, 15), [0])]
idx 1 cls_clicks: [((40, 50, 60), [1]), ((35, 45, 55), [0]), ((30, 40, 50), [0])]


In [42]:
cls_clicks[0][1]

[1]

In [10]:
ckpt_path =  "/mnt/aperto/yin/sammed3d_ckpt/sam_model_loss_best.pth"
model = medim.create_model("SAM-Med3D",
                               pretrained=True,
                               checkpoint_path=ckpt_path)


out_dir = "/mnt/aperto/yin/sammed_3D/sample_20/res"
img, spacing, all_clicks, prev_pred = read_data(img, clicks)
final_pred = np.zeros_like(img, dtype=np.uint8)
for idx, cls_clicks in all_clicks.items():
        category_index = idx + 1
        pred_ori = prev_pred==category_index
        final_pred[pred_ori!=0] = category_index
        if (cls_clicks[-1][1][0] == 1):
            cls_gt = create_gt_arr(img.shape, cls_clicks[-1][0], category_index=category_index)
            # print(category_index, imgs.shape, spacing, cls_clicks, (cls_gt==category_index).sum())
            # continue
            cls_prev_seg = prev_pred==category_index
            roi_image, roi_label, roi_prev_seg, meta_info = data_preprocess(img, cls_gt, cls_prev_seg,
                                                            orig_spacing=spacing, 
                                                            category_index=category_index)

            ''' 3. infer with the pre-trained SAM-Med3D model '''
            roi_pred = sam_model_infer(model, roi_image, roi_gt=roi_label, prev_low_res_mask=roi_prev_seg)

            ''' 4. post-process and save the result '''
            pred_ori = data_postprocess(roi_pred, meta_info, out_dir)
            final_pred[pred_ori!=0] = category_index

output_path = osp.join(out_dir,'test.npz')
np.savez_compressed(output_path, segs=final_pred)
print("result saved to", output_path)

creating model SAM-Med3D
try to load pretrained weights from /mnt/aperto/yin/sammed3d_ckpt/sam_model_loss_best.pth
result saved to /mnt/aperto/yin/sammed_3D/sample_20/res/test.npz


In [5]:
import napari
import numpy as np
label = np.load('/mnt/aperto/yin/zencell_plugin/segmented_res.npy')


In [6]:
label.shape

(40, 1024, 1024)

In [7]:
viewer = napari.Viewer()
viewer.add_labels(label)

<Labels layer 'label' at 0x78ca2efa1400>

In [ ]:
import numpy as np
import napari

data = np.random.rand(64, 128, 128).astype(np.float32)
viewer = napari.Viewer()
viewer.add_image(data)
napari.run()


Traceback (most recent call last):
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/app/backends/_qt.py", line 928, in paintGL
    self._vispy_canvas.events.draw(region=None)
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/util/event.py", line 453, in __call__
    self._invoke_callback(cb, event)
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/util/event.py", line 471, in _invoke_callback
    _handle_exception(self.ignore_callback_errors,
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/util/event.py", line 469, in _invoke_callback
    cb(event)
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/scene/canvas.py", line 219, in on_draw
    self._draw_scene()
  File "/mnt/aperto/anaconda/envs/zen_sam/lib/python3.12/site-packages/vispy/scene/canvas.py", line 278, in _draw_scene
    self.draw_visual(self.scene)
  File "/mnt/aperto/anaconda/envs/zen_sam/

In [34]:

import numpy as np
img = np.load('/mnt/aperto/yin/sammed_3D/sample_20/img/test_img.npz')['imgs']
res = np.load('/mnt/aperto/yin/sammed_3D/sample_20/res/test_cp.npz')['segs']



In [35]:
import napari
viewer = napari.Viewer()
viewer.add_image(img, name='sample_20_img')
viewer.add_labels(res, name='sample_20_res')

<Labels layer 'sample_20_res' at 0x7f9eca1a8ce0>

In [24]:
medsam_seg_prob = np.load('/mnt/aperto/yin/zencell_plugin/src/SAM-Med3D/medsam_seg_prob.npy')
flow_z = np.load('/mnt/aperto/yin/zencell_plugin/src/SAM-Med3D/flow_z.npy')
flow_y = np.load('/mnt/aperto/yin/zencell_plugin/src/SAM-Med3D/flow_y.npy')
flow_x = np.load('/mnt/aperto/yin/zencell_plugin/src/SAM-Med3D/flow_x.npy')


In [27]:
medsam_seg_prob.shape

(1, 1, 32, 32, 32)

In [28]:
medsam_seg_prob.squeeze().shape

(32, 32, 32)

In [13]:
dP = np.zeros((3, 32, 32, 32), dtype=np.float32)
dP[0] = flow_z[0,0].astype(np.float32)
dP[1] = flow_y[0,0].astype(np.float32)
dP[2] = flow_x[0,0].astype(np.float32)
cellprob = medsam_seg_prob[0,0].astype(np.float32)

In [17]:
from cellpose.dynamics import compute_masks

mask = compute_masks(dP, cellprob, min_size=0, flow_threshold=None, cellprob_threshold = 0.5, do_3D=True)

In [29]:
mask.shape

(32, 32, 32)

In [18]:
viewer = napari.Viewer()
viewer.add_image(medsam_seg_prob, name='medsam_seg_prob')
viewer.add_image(flow_z, name='flow_z')
viewer.add_image(flow_y, name='flow_y')
viewer.add_image(flow_x, name='flow_x')

viewer.add_labels(mask, name='mask')

<Labels layer 'mask' at 0x7f9e2297c080>

In [27]:
viewer = napari.Viewer()
viewer.add_image(roi_image, name='sample_20_roi_upscale')
viewer.add_image(low_res_mask, name='sample_20_low_res_mask')
viewer.add_image(medsam_seg_prob, name='sample_20_medsam_seg_prob')

<Image layer 'sample_20_medsam_seg_prob' at 0x7e7da83efad0>

In [22]:
np.unique(res)


array([0, 1], dtype=uint8)